# Data preparation — justification for grouping target vehicles into PR / BTW

Evidence that the target (following) vehicle sub-classes can be merged into two behavioural
groups (PR, BTW) for **time headway** and **target speed**:

1. **KDE + ECDF** overlays and pairwise **Kolmogorov–Smirnov** tests (distributional shape).
2. **Kruskal–Wallis** omnibus across sub-classes, with **Dunn–Holm** post-hoc, plus a grouped
   PR-vs-BTW test.
3. **Effect sizes** — epsilon-squared (omnibus) and Cliff's delta (pairwise), with magnitude labels.
4. **Equivalence tests (TOST)** — to positively demonstrate that sub-classes *within* a group are
   equivalent (a non-significant difference alone does not justify merging; equivalence does).

The logic: sub-classes assigned to the **same** group should be **equivalent** (TOST passes,
Dunn non-significant, small Cliff's delta, overlapping KDE/ECDF), while **PR vs BTW** should
**differ** (significant, large effect, non-equivalent, separated curves).

**Configure the granular target column and the group mapping in Cell 1.** `data3.xlsx` holds
only the merged `V_Target`; point `RAW_TARGET_COL` at your sub-class column to run the full
justification.

In [1]:
# --- Cell 1: Imports, paths, CONFIGURATION ---
import os
import numpy as np
import pandas as pd
from scipy import stats

BASE     = r"D:\Headway"
DATA     = os.path.join(BASE, "data3.xlsx")   # or the raw file that has the sub-classes
TABLES   = os.path.join(BASE, "Tables")
GRAPHICS = os.path.join(BASE, "Graphics")
os.makedirs(TABLES, exist_ok=True); os.makedirs(GRAPHICS, exist_ok=True)

# ---- CONFIGURE THESE ----
# Granular target-class column to justify merging. If your sub-classes live in another
# column/file, set RAW_TARGET_COL (and DATA) accordingly. Defaults to the merged V_Target.
RAW_TARGET_COL = "V_Target"

# Map each granular sub-class -> intended group ("PR" or "BTW").
# EDIT this to your real sub-classes, e.g. {"Car":"PR","Microbus":"PR","CNG":"BTW", ...}
# Left as identity so the notebook runs on the merged column out-of-the-box.
GROUP_MAP = {"PR": "PR", "BTW": "BTW"}

# Metrics to test (display name -> column). Speed column auto-detected below.
SPEED_COL = None   # resolved in Cell 2
METRICS = {"Time headway (s)": "Time_Headway", "Target speed (km/h)": "__SPEED__"}

# Equivalence margins per metric (in the metric's own units). If None, a margin of
# 0.2 x (pooled SD) is used ("small" by Cohen) -- REPLACE with a domain-justified value
# for the manuscript (e.g. 0.3 s for headway, 3 km/h for speed).
EQUIV_MARGIN = {"Time headway (s)": None, "Target speed (km/h)": None}

ALPHA = 0.05

In [2]:
# --- Cell 2: Load, resolve columns, assign groups ---
df = pd.read_excel(DATA)
SPEED_COL = "Target_Speed_km/hr" if "Target_Speed_km/hr" in df.columns else "Subject_Speed_km/hr"
METRICS = {k: (SPEED_COL if v == "__SPEED__" else v) for k, v in METRICS.items()}

if RAW_TARGET_COL not in df.columns:
    raise ValueError(f"RAW_TARGET_COL '{RAW_TARGET_COL}' not in data columns: {list(df.columns)}")

classes = [c for c in df[RAW_TARGET_COL].dropna().unique() if c in GROUP_MAP]
unmapped = [c for c in df[RAW_TARGET_COL].dropna().unique() if c not in GROUP_MAP]
if unmapped:
    print("WARNING: these sub-classes are not in GROUP_MAP and will be ignored:", unmapped)

GROUP_OF = {c: GROUP_MAP[c] for c in classes}
n_within = sum(list(GROUP_OF.values()).count(g) > 1 for g in set(GROUP_OF.values()))
print(f"Target column: {RAW_TARGET_COL} | sub-classes: {classes}")
print("Group membership:")
for g in sorted(set(GROUP_OF.values())):
    members = [c for c in classes if GROUP_OF[c] == g]
    print(f"   {g}: {members}  (n={[int((df[RAW_TARGET_COL]==m).sum()) for m in members]})")
if not any(list(GROUP_OF.values()).count(g) > 1 for g in set(GROUP_OF.values())):
    print("\nNOTE: no group has >=2 sub-classes -> within-group equivalence is trivial.")
    print("      Point RAW_TARGET_COL at your granular sub-class column for the full merge test.")

Target column: V_Target | sub-classes: ['BTW', 'PR']
Group membership:
   BTW: ['BTW']  (n=[669])
   PR: ['PR']  (n=[229])

NOTE: no group has >=2 sub-classes -> within-group equivalence is trivial.
      Point RAW_TARGET_COL at your granular sub-class column for the full merge test.


In [3]:
# --- Cell 3: Statistical helper functions ---
def ecdf(x):
    xs = np.sort(x); return xs, np.arange(1, len(xs)+1)/len(xs)

def kde_curve(x, grid):
    if len(np.unique(x)) < 3 or len(x) < 5: return None
    return stats.gaussian_kde(x)(grid)

def kruskal_eps(groups):
    H, p = stats.kruskal(*groups); N = sum(len(g) for g in groups)
    eps2 = H/(N-1)
    mag = ("negligible" if eps2 < 0.01 else "small" if eps2 < 0.06 else
           "medium" if eps2 < 0.14 else "large")
    return H, p, eps2, mag, N

def dunn_holm(data_by_class, group_of):
    labels = list(data_by_class.keys())
    data = np.concatenate([np.asarray(data_by_class[l]) for l in labels])
    grp  = np.concatenate([[l]*len(data_by_class[l]) for l in labels])
    N = len(data); ranks = stats.rankdata(data)
    _, counts = np.unique(data, return_counts=True)
    ties = np.sum(counts**3 - counts)
    sigma2 = (N*(N+1)/12.0) - ties/(12.0*(N-1))
    meanrank = {l: ranks[grp == l].mean() for l in labels}
    n = {l: int((grp == l).sum()) for l in labels}
    rows = []
    for i in range(len(labels)):
        for j in range(i+1, len(labels)):
            a, b = labels[i], labels[j]
            se = np.sqrt(sigma2*(1.0/n[a] + 1.0/n[b]))
            z = (meanrank[a] - meanrank[b])/se
            p = 2*stats.norm.sf(abs(z))
            comp = "within" if group_of[a] == group_of[b] else "between"
            rows.append([a, b, comp, n[a], n[b], round(z, 3), p])
    # Holm step-down over all pairwise p-values
    ps = np.array([r[6] for r in rows]); m = len(ps); order = np.argsort(ps); prev = 0.0
    adj = np.empty(m)
    for rank_i, idx in enumerate(order):
        prev = max(prev, (m - rank_i)*ps[idx]); adj[idx] = min(prev, 1.0)
    for r, a in zip(rows, adj):
        r[6] = round(r[6], 4); r.append(round(a, 4))
        r.append("Yes" if a < ALPHA else "No")
    return rows  # [a,b,comp,n_a,n_b,z,p_raw,p_holm,significant]

def cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    d = float(np.sign(x[:, None] - y[None, :]).mean())
    a = abs(d)
    mag = ("negligible" if a < 0.147 else "small" if a < 0.33 else
           "medium" if a < 0.474 else "large")
    return round(d, 4), mag

def pairwise_ks(x, y):
    D, p = stats.ks_2samp(x, y); return round(D, 4), p

def tost(x, y, margin):
    x = np.asarray(x); y = np.asarray(y)
    mx, my = x.mean(), y.mean(); d = mx - my
    vx, vy = x.var(ddof=1)/len(x), y.var(ddof=1)/len(y)
    se = np.sqrt(vx + vy)
    dfw = (vx+vy)**2 / (vx**2/(len(x)-1) + vy**2/(len(y)-1))
    p_low = stats.t.sf((d + margin)/se, dfw)     # H0: d <= -margin
    p_up  = stats.t.cdf((d - margin)/se, dfw)     # H0: d >=  margin
    p_tost = max(p_low, p_up)
    return round(d, 4), round(margin, 4), round(p_low, 4), round(p_up, 4), round(p_tost, 4), \
           ("Yes" if p_tost < ALPHA else "No")

In [4]:
# --- Cell 4: Run all statistics per metric ---
omnibus_rows, grouped_rows, dunn_rows, ks_rows, tost_rows = [], [], [], [], []

for mname, mcol in METRICS.items():
    sub = df[[RAW_TARGET_COL, mcol]].dropna()
    sub = sub[sub[RAW_TARGET_COL].isin(classes)]
    data_by_class = {c: sub.loc[sub[RAW_TARGET_COL] == c, mcol].values for c in classes}
    margin = EQUIV_MARGIN[mname] if EQUIV_MARGIN.get(mname) else 0.2*sub[mcol].std()

    # 1) omnibus KW across sub-classes
    if len(classes) >= 2:
        H, p, eps2, mag, N = kruskal_eps(list(data_by_class.values()))
        omnibus_rows.append(dict(Metric=mname, k_classes=len(classes), N=N, H=round(H,3),
                                 p="<0.001" if p<1e-3 else round(p,4),
                                 epsilon2=round(eps2,4), magnitude=mag))

    # 2) grouped PR vs BTW (Mann-Whitney + Cliff + TOST)
    gsub = sub.copy(); gsub["_grp"] = gsub[RAW_TARGET_COL].map(GROUP_OF)
    grps = {g: gsub.loc[gsub["_grp"] == g, mcol].values for g in sorted(gsub["_grp"].dropna().unique())}
    if len(grps) == 2:
        (ga, gb) = list(grps.keys())
        U, pg = stats.mannwhitneyu(grps[ga], grps[gb], alternative="two-sided")
        cd, cmag = cliffs_delta(grps[ga], grps[gb])
        td, tm, pl, pu, pt, eq = tost(grps[ga], grps[gb], margin)
        grouped_rows.append(dict(Metric=mname, group_A=ga, group_B=gb,
                                 median_A=round(np.median(grps[ga]),3), median_B=round(np.median(grps[gb]),3),
                                 MWU_U=round(U,1), p="<0.001" if pg<1e-3 else round(pg,4),
                                 cliffs_delta=cd, delta_mag=cmag,
                                 TOST_margin=tm, TOST_p=pt, equivalent=eq))

    # 3) Dunn-Holm across sub-classes + Cliff's delta per pair
    if len(classes) >= 2:
        for r in dunn_holm(data_by_class, GROUP_OF):
            a, b, comp, na, nb, z, p_raw, p_holm, sig = r
            cd, cmag = cliffs_delta(data_by_class[a], data_by_class[b])
            dunn_rows.append(dict(Metric=mname, class_A=a, class_B=b, comparison=comp,
                                  n_A=na, n_B=nb, z=z, p_raw=p_raw, p_holm=p_holm,
                                  significant=sig, cliffs_delta=cd, delta_mag=cmag))

    # 4) pairwise KS + 5) TOST equivalence per sub-class pair
    for i in range(len(classes)):
        for j in range(i+1, len(classes)):
            a, b = classes[i], classes[j]
            comp = "within" if GROUP_OF[a] == GROUP_OF[b] else "between"
            D, pk = pairwise_ks(data_by_class[a], data_by_class[b])
            ks_rows.append(dict(Metric=mname, class_A=a, class_B=b, comparison=comp,
                                KS_D=D, KS_p="<0.001" if pk<1e-3 else round(pk,4)))
            td, tm, pl, pu, pt, eq = tost(data_by_class[a], data_by_class[b], margin)
            tost_rows.append(dict(Metric=mname, class_A=a, class_B=b, comparison=comp,
                                  mean_diff=td, TOST_margin=tm, p_lower=pl, p_upper=pu,
                                  p_TOST=pt, equivalent=eq))

omnibus_kw   = pd.DataFrame(omnibus_rows)
grouped_test = pd.DataFrame(grouped_rows)
dunn_posthoc = pd.DataFrame(dunn_rows)
ks_matrix    = pd.DataFrame(ks_rows)
equivalence  = pd.DataFrame(tost_rows)
print("Omnibus KW:\n", omnibus_kw.to_string(index=False) if len(omnibus_kw) else "(needs >=2 sub-classes)")
grouped_test

Omnibus KW:
              Metric  k_classes   N       H      p  epsilon2 magnitude
   Time headway (s)          2 898 115.100 <0.001    0.1283    medium
Target speed (km/h)          2 898 118.535 <0.001    0.1321    medium


,Metric,group_A,group_B,median_A,median_B,MWU_U,p,cliffs_delta,delta_mag,TOST_margin,TOST_p,equivalent
0,Time headway (s),BTW,PR,1.90,3.0,40255.0,<0.001,-0.4745,large,0.1977,1.0,No
1,Target speed (km/h),BTW,PR,14.94,10.8,113484.0,<0.001,0.4815,large,1.0760,1.0,No


In [5]:
# --- Cell 5: Save all statistical tables -> Excel ---
out_path = os.path.join(TABLES, "prep_target_grouping.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    (omnibus_kw if len(omnibus_kw) else pd.DataFrame({"note":["needs >=2 sub-classes"]})
     ).to_excel(xl, sheet_name="Omnibus_KW", index=False)
    (grouped_test if len(grouped_test) else pd.DataFrame({"note":["PR/BTW grouping not resolved"]})
     ).to_excel(xl, sheet_name="Grouped_PR_vs_BTW", index=False)
    (dunn_posthoc if len(dunn_posthoc) else pd.DataFrame({"note":["needs >=2 sub-classes"]})
     ).to_excel(xl, sheet_name="Dunn_Holm_posthoc", index=False)
    ks_matrix.to_excel(xl,   sheet_name="Pairwise_KS", index=False)
    equivalence.to_excel(xl, sheet_name="Equivalence_TOST", index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\prep_target_grouping.xlsx


In [6]:
# --- Cell 6: KDE + ECDF data and native Excel charts (matplotlib-free) ---
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series

wb = load_workbook(out_path)
for mname, mcol in METRICS.items():
    sub = df[[RAW_TARGET_COL, mcol]].dropna(); sub = sub[sub[RAW_TARGET_COL].isin(classes)]
    lo, hi = sub[mcol].min(), sub[mcol].max(); grid = np.linspace(lo, hi, 200)
    ws = wb.create_sheet(("curves_" + mname[:20]).replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")[:31])
    ws.cell(1, 1, "grid"); [ws.cell(i+2, 1, float(grid[i])) for i in range(len(grid))]
    kde_ch = ScatterChart(); kde_ch.title = f"KDE — {mname}"; kde_ch.x_axis.delete=False; kde_ch.y_axis.delete=False
    ecdf_ch = ScatterChart(); ecdf_ch.title = f"ECDF — {mname}"; ecdf_ch.x_axis.delete=False; ecdf_ch.y_axis.delete=False
    col = 2
    for c in classes:
        x = sub.loc[sub[RAW_TARGET_COL] == c, mcol].values
        k = kde_curve(x, grid)
        if k is not None:
            ws.cell(1, col, f"kde_{c}")
            for i in range(len(grid)): ws.cell(i+2, col, float(k[i]))
            s = Series(Reference(ws, min_col=col, min_row=1, max_row=len(grid)+1),
                       Reference(ws, min_col=1, min_row=2, max_row=len(grid)+1), title_from_data=True)
            s.smooth = True; kde_ch.series.append(s); col += 1
        xs, ys = ecdf(x); base = col
        ws.cell(1, col, f"ecdf_x_{c}"); ws.cell(1, col+1, f"ecdf_y_{c}")
        for i in range(len(xs)): ws.cell(i+2, col, float(xs[i])); ws.cell(i+2, col+1, float(ys[i]))
        s2 = Series(Reference(ws, min_col=col+1, min_row=1, max_row=len(xs)+1),
                    Reference(ws, min_col=col, min_row=2, max_row=len(xs)+1), title_from_data=True)
        s2.smooth = False; ecdf_ch.series.append(s2); col += 2
    kde_ch.height, kde_ch.width = 9, 15; ecdf_ch.height, ecdf_ch.width = 9, 15
    ws.add_chart(kde_ch, "B25"); ws.add_chart(ecdf_ch, "B45")
wb.save(out_path)
print("Embedded KDE/ECDF charts into:", out_path)

Embedded KDE/ECDF charts into: D:\Headway\Tables\prep_target_grouping.xlsx


In [7]:
# --- Cell 7: Publication KDE/ECDF overlays via matplotlib (skipped if blocked) ---
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
                         "figure.dpi": 300, "savefig.bbox": "tight"})
    group_colour = {"PR": "#c0392b", "BTW": "#2c3e50"}
    for mname, mcol in METRICS.items():
        sub = df[[RAW_TARGET_COL, mcol]].dropna(); sub = sub[sub[RAW_TARGET_COL].isin(classes)]
        lo, hi = sub[mcol].min(), sub[mcol].max(); grid = np.linspace(lo, hi, 200)
        fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.8))
        for c in classes:
            x = sub.loc[sub[RAW_TARGET_COL] == c, mcol].values
            colr = group_colour.get(GROUP_OF[c], None)
            k = kde_curve(x, grid)
            if k is not None:
                ax[0].plot(grid, k, color=colr, lw=1.6, label=f"{c} ({GROUP_OF[c]})")
            xs, ys = ecdf(x); ax[1].step(xs, ys, where="post", color=colr, lw=1.4, label=f"{c} ({GROUP_OF[c]})")
        ax[0].set_xlabel(mname); ax[0].set_ylabel("Density"); ax[0].legend(frameon=False, fontsize=8)
        ax[1].set_xlabel(mname); ax[1].set_ylabel("F(x)"); ax[1].legend(frameon=False, fontsize=8, loc="lower right")
        fig.suptitle(f"Target sub-classes — {mname}", fontsize=11)
        fp = os.path.join(GRAPHICS, f"prep_target_{mcol}.png".replace("/", "_"))
        fig.savefig(fp); plt.close(fig)
    print("Saved KDE/ECDF PNGs to:", GRAPHICS)
except Exception as e:
    print("matplotlib unavailable (", type(e).__name__, ") - skipped PNGs; use the Excel charts /")
    print("exported curve data in prep_target_grouping.xlsx instead.")

Saved KDE/ECDF PNGs to: D:\Headway\Graphics
